# RCS MARBERTv2 Training — Google Colab

Train+val only (`data/gold/gold_train_val_only.csv`) — this is the human_test-excluded
export of the canonical `data/gold/gold_dataset_final_training.csv` (455 supervised
train rows / 108 supervised val rows, 5-class Intent taxonomy with DUA_BLESSING merged
into THANKS_PRAISE). The raw canonical file, which contains the 75 sealed `human_test`
rows, is never packaged into this bundle. Cell 1 verifies this provenance at runtime
against `data/gold/bundle_training_manifest.json` rather than asking you to trust the
filename. **No human_test.**

1. Upload `rcs_colab_training_bundle.zip` to `/content/`
2. Run cells in order
3. **Restart runtime** after install
4. Preflight must pass before training


In [ ]:
# Cell 1 — Extract bundle + verify gold CSV provenance
import hashlib
import json
import os
from pathlib import Path

ZIP = Path("/content/rcs_colab_training_bundle.zip")
if not ZIP.exists():
    raise FileNotFoundError("Upload rcs_colab_training_bundle.zip to /content/ first")

!unzip -q -o {ZIP} -d /content/
WORK = Path("/content/rcs_project")
os.chdir(WORK)
os.environ["PYTHONPATH"] = "src:."
os.environ["HF_HOME"] = "/content/hf_cache"
# GOLD_CSV is the train+val-only export of the canonical
# data/gold/gold_dataset_final_training.csv (human_test excluded before
# packaging -- the raw canonical is never shipped into this bundle).
GOLD_CSV = "data/gold/gold_train_val_only.csv"

manifest = json.loads(Path("data/gold/bundle_training_manifest.json").read_text())
actual_sha256 = hashlib.sha256(Path(GOLD_CSV).read_bytes()).hexdigest()
print("Project root:", WORK)
print("GOLD_CSV:", GOLD_CSV)
print("source canonical (gold_dataset_final_training.csv) sha256:", manifest["source_canonical_gold_sha256"])
print("GOLD_CSV sha256 match vs manifest:", actual_sha256 == manifest["train_val_csv_sha256"])
print("supervised_train_rows:", manifest["supervised_train_rows"], "| supervised_val_rows:", manifest["supervised_val_rows"], "| human_test_rows_excluded:", manifest["human_test_rows_excluded"])
assert actual_sha256 == manifest["train_val_csv_sha256"], "GOLD_CSV does not match the bundle manifest -- do not proceed"
assert manifest["supervised_train_rows"] == 455, manifest["supervised_train_rows"]
assert manifest["supervised_val_rows"] == 108, manifest["supervised_val_rows"]
assert manifest["human_test_rows_excluded"] == 75, manifest["human_test_rows_excluded"]


In [ ]:
# Cell 2 — Install GPU-compatible PyTorch + ML stack
# REQUIRED: Restart runtime after this cell (Runtime → Restart runtime)
!python colab/setup_gpu_env.py --install

## Restart runtime now

**Runtime → Restart runtime**, then continue from Cell 3.

In [ ]:
# Cell 3 — Post-restart environment restore + re-verify provenance
import hashlib
import json
import os
from pathlib import Path

WORK = Path("/content/rcs_project")
os.chdir(WORK)
os.environ["PYTHONPATH"] = "src:."
os.environ["HF_HOME"] = "/content/hf_cache"
GOLD_CSV = "data/gold/gold_train_val_only.csv"

manifest = json.loads(Path("data/gold/bundle_training_manifest.json").read_text())
actual_sha256 = hashlib.sha256(Path(GOLD_CSV).read_bytes()).hexdigest()
assert actual_sha256 == manifest["train_val_csv_sha256"], "GOLD_CSV does not match the bundle manifest -- do not proceed"
assert manifest["supervised_train_rows"] == 455, manifest["supervised_train_rows"]
assert manifest["supervised_val_rows"] == 108, manifest["supervised_val_rows"]
print("Provenance re-verified after restart: train=", manifest["supervised_train_rows"], "val=", manifest["supervised_val_rows"])


In [ ]:
# Cell 4 — CUDA verify + strict preflight + MARBERTv2 smoke
!python colab/setup_gpu_env.py --verify
!python colab/preflight.py --gold-csv {GOLD_CSV}

In [ ]:
# Cell 5 — Data preflight manifest (no training)
!python scripts/prepare_training_data.py --gold-csv {GOLD_CSV}

In [ ]:
# Cell 6 -- Full training (run only after preflight passes)
# 2026-08-24: --skip-classical removed so TF-IDF Intent/Topic/Sentiment
# baselines are also freshly fit on the expanded train/val split, for a
# direct comparison against the previously registered TF-IDF results in
# TRAINING_RESULTS_AUDIT.md (not just the MARBERTv2 numbers).
# --skip-sentiment added: this runtime is pinned to torch 2.5.1+cu121
# (see colab/setup_gpu_env.py). Pretrained CAMeLBERT sentiment inference
# needs torch>=2.6 and runs separately in colab/phase_e_sentiment_colab.ipynb
# -- run that notebook too, in its own fresh runtime, for sentiment results.
OUTPUT = "/content/selection"
!python scripts/run_model_selection.py \
  --gold-csv {GOLD_CSV} \
  --output-dir {OUTPUT} \
  --skip-sentiment

In [ ]:
# Cell 7 — Package artifacts for download
!python colab/package_artifacts.py --selection-dir /content/selection
from google.colab import files
files.download("/content/rcs_selection_artifacts.zip")

In [ ]:
# Optional — copy artifacts to Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/rcs_selection
# !rsync -a /content/selection/ /content/drive/MyDrive/rcs_selection/